Test all the functions one by one

In [9]:
import Rolling_Intrinsic_QH as RI
import os
import pandas as pd

In [ ]:
path = os.path.join("..","..","fake_data","transaction_data.parquet")
df = RI.load_fake_data(path)

In [21]:
df.set_index("executiontime",inplace=True)

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 6000000 entries, 2022-01-01 00:00:00+01:00 to 2023-01-02 00:45:00+01:00
Data columns (total 6 columns):
 #   Column         Dtype                        
---  ------         -----                        
 0   deliverystart  datetime64[ns, Europe/Berlin]
 1   deliveryend    datetime64[ns, Europe/Berlin]
 2   price          float32                      
 3   volume         float32                      
 4   side           object                       
 5   product        object                       
dtypes: datetime64[ns, Europe/Berlin](2), float32(2), object(2)
memory usage: 306.9+ MB


In [121]:
def get_average_prices(
    df, side, execution_time_start, execution_time_end, end_date, min_trades=10
    ):
    
    # set start_of_day to end_date minus 1 day
    start_of_day = pd.to_datetime(end_date).tz_localize('Europe/Berlin') - pd.Timedelta(hours=2)

    # set hour and minute to 0 (europe/berlin time)
    start_of_day = start_of_day.replace(hour=0, minute=0)

    end_of_day = start_of_day

    end_of_day = end_of_day.replace(hour=23, minute=45)
    # This is the Postgres query that checks for transactions:
    #   within a certain timestep (Rolling Window),
    #   Where products are:  
    #       either buy or sell transactions,
    #       With delivery within the chosen day
    #       The results are grouped by product, and only groups with over the threshold of min trades are kept fetched
    # It returns volume wighted average price

    # cursor.execute(f"""
    #     SELECT
    #     deliverystart,
    #     SUM(price*volume)/SUM(volume) AS weighted_avg_price
    #     FROM
    #     transactions_intraday_de
    #     WHERE
    #     (executiontime BETWEEN '{execution_time_start}' AND '{execution_time_end}')
    #     AND (product ='XBID_Quarter_Hour_Power' or product = 'Intraday_Quarter_Hour_Power') AND side='{side}' AND deliverystart < '{end_date}' AND deliverystart >= '{start_of_day}'
    #     GROUP BY
    #     deliverystart
    #     HAVING
    #     COUNT(*) >= {min_trades};
    #     """)
    # result = cursor.fetchall()
    df_bucket = df.loc[execution_time_start:execution_time_end,:].copy()
    
    filter = (df_bucket.side==side) & (df_bucket.deliverystart < end_date) & (df_bucket.deliverystart>=start_of_day)
    df_bucket = df_bucket[filter]
    #continuehere
    result = df_bucket.groupby("deliverystart").filter(lambda x: len(x)>=min_trades)
    result = result.groupby("deliverystart")\
          .apply(func = (lambda x: (x.price * x.volume).sum() / x.volume.sum())).reset_index()
    #result = VWAP from bucket
    df_vwap = pd.DataFrame(result.values, columns=["product", "price"])
    df_vwap = df_vwap.astype(
       { "price" : "float64",
       }
    )
   
    # set index to product
    df_vwap.set_index("product", inplace=True)
    #print(df_vwap) 
    # set index to be all 15 minute intervals from start_of_day to end_of_day, filling missing values with NaN
    df_vwap = df_vwap.reindex(pd.date_range(start_of_day, end_of_day, freq="15min"))

    return df_vwap

In [122]:
vwap = get_average_prices(
                df,
                side="BUY",
                execution_time_start="2022-01-02 00:00:00",
                execution_time_end="2022-01-02 00:00:15",
                end_date = "2022-01-03 00:00:00",
                min_trades=2,
            )
vwap


,price
2022-01-02 00:00:00+01:00,NaN
2022-01-02 00:15:00+01:00,NaN
2022-01-02 00:30:00+01:00,67.369446
2022-01-02 00:45:00+01:00,NaN
2022-01-02 01:00:00+01:00,53.040497
...,...
2022-01-02 22:45:00+01:00,NaN
2022-01-02 23:00:00+01:00,NaN
2022-01-02 23:15:00+01:00,NaN
2022-01-02 23:30:00+01:00,NaN
